# 📓 2. From Zero to a Working DBT
### 🎯 Interactive Workshop Notebook

**Welcome!** In this notebook, you’ll turn your tool setup into a **working DBT project**.

## What you’ll do
- ✅ Check that you opened the notebook in the right repo
- ✅ Create a project virtual environment
- ✅ Install the workshop packages
- ✅ Connect DBT to DuckDB
- ✅ Load data and run models
- ✅ Query the results you created

## How to use this notebook
1. Read the short explanation
2. Run the cell below it ▶️
3. Check the result against the checkpoint ✅
4. If something breaks, use the troubleshooting section at the end

> **Good news:** this notebook avoids tricky OS-specific commands where possible.
> It uses Python paths directly, so it works more reliably on **Windows, Linux, and Chromebook Linux**.


## 🔍 Step 0 — Check your setup and find the repo

This cell does three important things:
- shows your Python + operating system
- finds the **repo root** automatically
- checks that the `dbt_project/` folder exists


In [1]:
from pathlib import Path
import os, platform, re, subprocess, sys, textwrap

OS_NAME = platform.system()
print(f"Python: {sys.version.split()[0]}")
print(f"Operating System: {OS_NAME}")

def find_repo_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'README.md').exists() and (candidate / 'dbt_project').exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the workshop repo root. Open this notebook from inside the cloned dbt-workshop repo."
    )

repo_root = find_repo_root(Path.cwd())
project_dir = repo_root / 'dbt_project'
venv_dir = repo_root / '.venv'
venv_python = venv_dir / ('Scripts/python.exe' if OS_NAME == 'Windows' else 'bin/python')
venv_pip = venv_dir / ('Scripts/pip.exe' if OS_NAME == 'Windows' else 'bin/pip')
venv_dbt = venv_dir / ('Scripts/dbt.exe' if OS_NAME == 'Windows' else 'bin/dbt')
requirements_file = repo_root / ('requirements_locked.txt' if (repo_root / 'requirements_locked.txt').exists() else 'requirements.txt')

print(f"Repo root: {repo_root}")
print(f"DBT project folder found: {project_dir.exists()}")
print(f"Requirements file: {requirements_file.name}")


Python: 3.11.2
Operating System: Linux


FileNotFoundError: Could not find the workshop repo root. Open this notebook from inside the cloned dbt-workshop repo.

### ✅ Checkpoint
You should see:
- a Python version
- your operating system
- the path to your repo root
- `DBT project folder found: True`

If you do **not** see that last line as `True`, stop here and open the notebook from the correct repo.


## 🧪 Step 1 — Create a virtual environment

A virtual environment keeps your project neat.
It means your workshop tools stay together instead of getting mixed into your whole computer.


In [ ]:
import venv

if venv_python.exists():
    print('✅ Virtual environment already exists')
else:
    print('Creating virtual environment...')
    venv.create(venv_dir, with_pip=True)
    print(f'✅ Created: {venv_dir}')

print(f'Python inside environment: {venv_python}')


### ✅ Checkpoint
You should see either:
- `Virtual environment already exists`
or
- `Created: .../.venv`


## 📦 Step 2 — Install the workshop packages

This installs everything needed for the workshop.

> We use `requirements_locked.txt` when available because it gives a more stable conference setup.
> That means fewer surprise version problems.


In [ ]:
def run_command(cmd, cwd=None, check=True):
    print('Running:', ' '.join(str(x) for x in cmd))
    completed = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.returncode != 0 and completed.stderr.strip():
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')
    return completed

run_command([venv_python, '-m', 'pip', 'install', '--upgrade', 'pip'], cwd=repo_root)
run_command([venv_pip, 'install', '-r', requirements_file], cwd=repo_root)
print('✅ Packages installed')


### ✅ Checkpoint
You should see:
- `Packages installed`
- package names including **dbt-duckdb** and **duckdb** somewhere in the install output


## 🔎 Step 3 — Validate the main tools

Let’s make sure the virtual environment can actually run:
- Python
- pip
- dbt


In [ ]:
run_command([venv_python, '--version'])
run_command([venv_pip, '--version'])
run_command([venv_dbt, '--version'])
print('✅ Core tools are working')


### ✅ Checkpoint
You should see version information for all three tools.

> Seeing a message about a newer DBT version is **not** a problem here.
> For a workshop, consistency is more important than always being newest.


## ⚙️ Step 4 — Create a DBT profile automatically

DBT needs a small config file called `profiles.yml`.
This tells DBT how to connect to your database.

In this workshop, DBT will connect to a local **DuckDB** file inside the project folder.


In [ ]:
dbt_project_yml = project_dir / 'dbt_project.yml'
if not dbt_project_yml.exists():
    raise FileNotFoundError(f'Could not find {dbt_project_yml}')

project_name = None
for line in dbt_project_yml.read_text(encoding='utf-8').splitlines():
    match = re.match(r'^\s*name\s*:\s*["\']?([^"\']+)["\']?\s*$', line)
    if match:
        project_name = match.group(1).strip()
        break

if not project_name:
    raise ValueError('Could not read the project name from dbt_project/dbt_project.yml')

profiles_dir = Path.home() / '.dbt'
profiles_dir.mkdir(exist_ok=True)
database_path = (project_dir / 'workshop.duckdb').resolve()

profile_text = f'''{project_name}:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: "{database_path.as_posix()}"
      threads: 1
'''

profile_file = profiles_dir / 'profiles.yml'
profile_file.write_text(profile_text, encoding='utf-8')

print('✅ Profile created')
print(f'Project name: {project_name}')
print(f'Profile file: {profile_file}')
print(f'Database file: {database_path}')


### ✅ Checkpoint
You should see:
- `Profile created`
- your DBT project name
- a DuckDB file path ending in `workshop.duckdb`


## 🧪 Step 5 — Test the DBT connection

Now we check whether:
- the profile is valid
- the project is valid
- DBT can connect to DuckDB


In [ ]:
debug_result = run_command([
    venv_dbt,
    'debug',
    '--project-dir', project_dir,
    '--profiles-dir', profiles_dir,
], cwd=repo_root, check=False)

if debug_result.returncode == 0:
    print('✅ DBT debug passed')
else:
    raise RuntimeError('DBT debug failed. Scroll up and read the output carefully.')


### ✅ Checkpoint
You want to see a successful DBT debug result.

The most important signs are:
- the profile is found
- the project is found
- the connection is OK


## 🌱 Step 6 — Load the seed data

A **seed** is just a CSV file that DBT loads into the database for you.

In this workshop, that means your raw data gets pulled into DuckDB, ready for modelling.


In [ ]:
seed_result = run_command([
    venv_dbt,
    'seed',
    '--project-dir', project_dir,
    '--profiles-dir', profiles_dir,
], cwd=repo_root, check=False)

if seed_result.returncode == 0:
    print('✅ Seed completed successfully')
else:
    raise RuntimeError('DBT seed failed. Scroll up and read the output carefully.')


### ✅ Checkpoint
You should see a success message for the seed step.

That means the CSV data is now inside your DuckDB database.


## 🧱 Step 7 — Run the transformation models

This is the big moment.

DBT now takes the raw data and builds cleaner, more useful tables from it.


In [ ]:
run_result = run_command([
    venv_dbt,
    'run',
    '--project-dir', project_dir,
    '--profiles-dir', profiles_dir,
], cwd=repo_root, check=False)

if run_result.returncode == 0:
    print('✅ DBT run completed successfully')
else:
    raise RuntimeError('DBT run failed. Scroll up and read the output carefully.')


### ✅ Checkpoint
You should see success output from DBT.

Depending on your project, you should now have transformed models such as:
- `customer_summary`
- `high_value_customers`


## 🧪 Step 8 — Run DBT tests (recommended)

This checks whether your project rules still hold.

Think of tests like a quick quality check before you trust the data.


In [ ]:
test_result = run_command([
    venv_dbt,
    'test',
    '--project-dir', project_dir,
    '--profiles-dir', profiles_dir,
], cwd=repo_root, check=False)

if test_result.returncode == 0:
    print('✅ DBT tests passed')
else:
    print('⚠️ Some tests failed. That does not always mean your install is broken, but it is worth checking.')


## 🔍 Step 9 — Explore `customer_summary`

Now let’s query the table you created.

> This cell uses the Python inside your virtual environment, so it works even if your notebook kernel is different.


In [ ]:
query_code = textwrap.dedent(f'''
from pathlib import Path
import duckdb

db_path = Path(r"{database_path}")
con = duckdb.connect(str(db_path))
df = con.execute("SELECT * FROM customer_summary LIMIT 5").fetchdf()
print(df.to_string(index=False))
''')

run_command([venv_python, '-c', query_code], cwd=repo_root)


### ✅ Checkpoint
You should see a small table of results.

Nice — you are now reading data from a database that **you** built.


## 💎 Step 10 — Explore `high_value_customers`

This is a nice example of how DBT turns raw data into a more useful business view.


In [ ]:
query_code = textwrap.dedent(f'''
from pathlib import Path
import duckdb

db_path = Path(r"{database_path}")
con = duckdb.connect(str(db_path))
df = con.execute("SELECT * FROM high_value_customers").fetchdf()
print(df.to_string(index=False))
''')

run_command([venv_python, '-c', query_code], cwd=repo_root)


### ✅ Checkpoint
You should see a filtered list of customers.


## 🔥 Mini challenge

Open one of your SQL model files and make a small change.

Good options:
- `dbt_project/models/marts/customer_summary.sql`
- `dbt_project/models/marts/high_value_customers.sql`

Then rerun the next cell and see what changes in the output.


In [ ]:
run_command([
    venv_dbt,
    'run',
    '--project-dir', project_dir,
    '--profiles-dir', profiles_dir,
], cwd=repo_root)


## 🛠️ Troubleshooting

### Problem: `dbt` is not found
Run Step 2 again. That usually means the package install did not finish properly.

### Problem: `DBT debug failed`
Check these carefully:
- are you inside the correct repo?
- does `dbt_project/dbt_project.yml` exist?
- was `profiles.yml` created in your home `.dbt/` folder?

### Problem: permission errors on Linux or Chromebook
Close the notebook, open a fresh terminal, and run Jupyter again from your own user account.

### Problem: you want to rebuild from scratch
Run the next cell to remove the virtual environment, then go back to Step 1.

> Only use the reset cell if you really want a clean restart.


In [ ]:
import shutil

if venv_dir.exists():
    shutil.rmtree(venv_dir)
    print(f'🧹 Removed: {venv_dir}')
else:
    print('No virtual environment found to remove.')


## 🧠 What you achieved

You just:
- created a virtual environment ✅
- installed real project tools ✅
- configured DBT ✅
- loaded data into DuckDB ✅
- ran transformations ✅
- queried your results ✅

That is real data workflow stuff — not just theory.


## 🎉 You did it

You now have a working DBT setup and a working local data pipeline.

## ✅ Next step
Open the project notebook in:

`03_project/From_Data_to_Decision.ipynb`

That is where you will:
- explore patterns 📊
- build insights 💡
- turn data into decisions 🚀
